In [1]:
import librosa as lr
import numpy as np
import os
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [2]:
# --- MODEL PARAMS ---

JUKEBOX_SAMPLE_RATE = 44100
T = 8192
SAMPLE_LENGTH = 1048576  # ~23.77s, which is the param in JukeMIR
DEPTH = 36

In [3]:
def load_audio_from_file(fpath):
    audio, _ = lr.load(fpath, sr=JUKEBOX_SAMPLE_RATE)
    if audio.ndim == 1:
        audio = audio[np.newaxis]
    audio = audio.mean(axis=0)

    # normalize audio
    norm_factor = np.abs(audio).max()
    if norm_factor > 0:
        audio /= norm_factor

    return audio.flatten()

In [4]:
def split_audio_into_four_random_parts(audio, chunk_size=SAMPLE_LENGTH, sample_rate=JUKEBOX_SAMPLE_RATE):
    """
    Splits the audio into four equal parts and randomly samples `chunk_size` from each part,
    ensuring that the sampled chunk does not exceed the part's boundary.

    Args:
    - audio (numpy.ndarray): The loaded and preprocessed audio signal.
    - chunk_size (int): Size of each sample chunk in number of samples.
    - sample_rate (int): Sampling rate of the audio. Default is 44100 Hz.

    Returns:
    - List of four audio chunks (numpy.ndarray), each randomly sampled from one of the four equal parts of the audio.
    """
    
    num_samples = audio.shape[0]
    part_size = num_samples // 4  # Divide the audio into four equal parts
    
    chunks = []
    for i in range(4):
        # Calculate start and end samples for each part
        start_of_part = i * part_size
        end_of_part = start_of_part + part_size
        
        # Ensure the sampled chunk does not exceed the part's boundary
        max_start_sample = end_of_part - chunk_size
        
        # Randomly choose start sample within the part
        start_sample = np.random.randint(start_of_part, max_start_sample)
        chunk = audio[start_sample:start_sample + chunk_size]
           
        
        chunks.append(chunk)
    
    return chunks

In [5]:
def process_audio_file(filename, audio_path):
    try:
        audio = load_audio_from_file(audio_path)
        audio_chunks = split_audio_into_four_random_parts(audio)
    except:
        print(filename)
    return filename, audio_chunks

In [6]:
# For Test

audio_path = "/Users/yeongho/Library/CloudStorage/GoogleDrive-dslmodelingrecsys@gmail.com/내 드라이브/datasets/music/378.wav"

audio = load_audio_from_file(audio_path)
print("Audio Shape : ", audio.shape)

audio_chunks = split_audio_into_four_random_parts(audio)
print("Length of Audio Chunks : ", len(audio_chunks))
print("Audio Chunk Shape 1: ", audio_chunks[0].shape)
print("Audio Chunk Shape 2: ", audio_chunks[1].shape)
print("Audio Chunk Shape 3: ", audio_chunks[2].shape)
print("Audio Chunk Shape 4: ", audio_chunks[3].shape)

Audio Shape :  (7862273,)
Length of Audio Chunks :  4
Audio Chunk Shape 1:  (1048576,)
Audio Chunk Shape 2:  (1048576,)
Audio Chunk Shape 3:  (1048576,)
Audio Chunk Shape 4:  (1048576,)


In [13]:
base_path = "/Users/yeongho/Library/CloudStorage/GoogleDrive-dslmodelingrecsys@gmail.com/내 드라이브/datasets/music"
file_info = {}
# 377 Newjeans - Get up 
# 454 TripleS - Before the Rise 
tooshort = [377, 454]

for i in range(1300, 1900):
    if i not in tooshort:
        file_path = os.path.join(base_path, f"{i}.wav")
        file_info[i] = file_path

# 멀티프로세싱을 사용하여 모든 파일 처리
num_cpus = os.cpu_count()-2

with ThreadPoolExecutor(max_workers = num_cpus) as executor:
    results = {}
    futures = [executor.submit(process_audio_file, id, file_path) for id, file_path in file_info.items()]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Files"):
        result = future.result()
        key = int(result[0])
        results[key] = result[1]

Processing Files: 100%|██████████| 600/600 [00:53<00:00, 11.29it/s]


In [14]:
with open('/Users/yeongho/Library/CloudStorage/GoogleDrive-dslmodelingrecsys@gmail.com/내 드라이브/datasets/music_lr/1300_1900.pkl', 'wb') as f:
      pickle.dump(results, f)

In [3]:
sample3s = {}
filelist = [0, 100, 200, 300, 400, 500, 600, 700, 1300, 1900, 2076]
for i, file in enumerate(filelist):
    if i != 10:
        path = f'/Users/yeongho/Library/CloudStorage/GoogleDrive-dslmodelingrecsys@gmail.com/내 드라이브/datasets/music_lr/{file}_{filelist[i+1]}.pkl'            
        with open(path, 'rb') as f:
            pkl = pickle.load(f)
        for key, value in pkl.items():
            sample3s[key] = value


In [4]:
with open('/Users/yeongho/Desktop/Modeling/sample3s.pkl', 'wb') as f:
      pickle.dump(sample3s, f)

: 